In [1]:
import torch
from torchvision.datasets import ImageFolder
from torchvision import transforms


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
print(device)

cuda


In [4]:
dataset = ImageFolder("train", transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
]))

In [5]:
print(len(dataset))

4750


In [6]:
print(dataset.class_to_idx)

{'Black-grass': 0, 'Charlock': 1, 'Cleavers': 2, 'Common Chickweed': 3, 'Common wheat': 4, 'Fat Hen': 5, 'Loose Silky-bent': 6, 'Maize': 7, 'Scentless Mayweed': 8, 'Shepherds Purse': 9, 'Small-flowered Cranesbill': 10, 'Sugar beet': 11}


In [7]:
from sklearn.model_selection import train_test_split


In [8]:
targets = dataset.targets
idx = list(range(len(dataset)))

train_idx, temp_idx = train_test_split(idx , test_size = 0.3, stratify = targets , random_state =42)

In [9]:
val_idx, test_idx = train_test_split(temp_idx, test_size = 0.5 , stratify = [targets[i] for i in temp_idx], random_state =42)

In [10]:
print(len(train_idx) , len(val_idx), len(test_idx))

3325 712 713


In [11]:
import torch.nn as nn

class Net(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.feature_extractor = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size =3 , padding = 1),
            nn.ELU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(32, 64, kernel_size=3, padding = 1),
            nn.ELU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Flatten())
        self.classifier = nn.Linear(64 * 32 * 32 , num_classes)
    
    def forward(self , x):
        x = self.feature_extractor(x)
        x = self.classifier(x)
        return x
        


In [11]:
from torch.utils.data import DataLoader , Subset

In [12]:
train_set = Subset(dataset , train_idx)
val_set = Subset(dataset , val_idx)
test_set = Subset(dataset, test_idx)

In [13]:
train_loader = DataLoader(train_set , batch_size = 4 , shuffle =True)
val_loader = DataLoader(val_set , batch_size = 4 , shuffle =False)
test_loader = DataLoader(test_set , batch_size = 4 , shuffle =False)

In [14]:
import torch.optim as optim

In [16]:
model = Net(num_classes = 12)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
model.train()
for epoch in range(10):
    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

In [21]:
from torchmetrics import Precision, Recall
precision = Precision(task = 'multiclass', num_classes = 12, average = 'weighted')
recall = Recall(task = 'multiclass', num_classes = 12, average = 'weighted')
model.eval()
with torch.no_grad():
    for images, labels in val_loader:
        outputs = model(images)
        _, preds = torch.max(outputs,1)
        precision(preds, labels)
        recall(preds,labels)
precision_score = precision.compute()
recall_score = recall.compute()

In [22]:
print(f"Precision: {precision_score:.4f}")
print(f"Recall: {recall_score:.4f}")

Precision: 0.4494
Recall: 0.4101


ResNet18

In [21]:
from torchvision import models
import torch.nn as nn

In [19]:
model = models.resnet18(weights = models.ResNet18_Weights.DEFAULT)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\Top Prix/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100.0%


In [22]:
model.fc = nn.Linear(model.fc.in_features, 12)
for param in model.parameters():
    param.requires_grad=False
for param in model.layer4.parameters():
    param.requires_grad = True
for param in model.fc.parameters():
    param.requires_grad=True

In [24]:
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.0001)

In [25]:
criterion = nn.CrossEntropyLoss()

In [27]:
for epoch in range(10):
    running_loss = 0
    for images, labels in train_loader:
        optimizer.zero_grad()
        output = model(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    epoch_loss = running_loss / len(train_loader)
    print(f"epoch : {epoch+1}, loss = {running_loss:.4f}")

epoch : 1, loss = 1167.5118
epoch : 2, loss = 611.1759
epoch : 3, loss = 419.8078
epoch : 4, loss = 324.4710
epoch : 5, loss = 234.5048
epoch : 6, loss = 218.9852
epoch : 7, loss = 178.1654
epoch : 8, loss = 154.7629
epoch : 9, loss = 154.3845
epoch : 10, loss = 141.8947


In [28]:
from torchmetrics import Recall , Precision
recall = Recall(task = 'multiclass' , num_classes = 12 , average = 'weighted')
precision = Precision(task = 'multiclass' , num_classes = 12 , average = 'weighted')


In [30]:

model.eval()
with torch.no_grad():
    for images , labels in val_loader:
        output = model(images)
        _ , pred = torch.max(output , 1)
        recall(pred , labels)
        precision(pred , labels)
recall_score = recall.compute()
precision_score = precision.compute()

In [31]:
print(f"Precision score is : {precision_score}")
print(f"recall score is: {recall_score}")

Precision score is : 0.8272581100463867
recall score is: 0.8216292262077332


In [32]:
torch.save(model.state_dict(), 'model_weights.pth')

In [38]:
print(train_set.class_to_idx_)

AttributeError: 'Subset' object has no attribute 'class_to_idx_'